In [1]:
import os
import json
import glob
import random
from typing import Any, Dict, List, Union
from pathlib import Path
from datasets import load_dataset
from dotenv import load_dotenv
from underthesea import word_tokenize

import asyncio
from typing import List, Dict, Any, Awaitable

from pydantic import BaseModel
from openai import AsyncOpenAI
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    Retrying,
    retry_if_exception_type,
    before_sleep_log,
    after_log,
)
import logging
import aiofiles

# Configure logging for the retry mechanism
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


load_dotenv()


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

In [2]:
from openai import AsyncOpenAI, OpenAI

aclient = AsyncOpenAI(
    # Replace "YOUR_GEMINI_API_KEY" with your actual Gemini API key
    api_key=GEMINI_API_KEY,
    # This base_url routes the request to the Gemini API's compatibility layer
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [3]:
# Define the type alias for clarity
JsonData = Union[Dict[str, Any], List[Any]]


def write_json(filepath: str, data: JsonData) -> bool:
    """
    Writes data to a specified JSON file path.

    Args:
        filepath: The path to the output JSON file.
        data: The Python data structure (dict or list) to save.

    Returns:
        True if the write was successful, False otherwise.
    """
    try:
        # Open file in write mode ('w'), specifying UTF-8 encoding
        with open(filepath, "w", encoding="utf-8") as f:
            # Use json.dump for writing. indent=4 makes the file human-readable.
            json.dump(data, f, indent=4)
        print(f"Successfully wrote data to {filepath}")
        return True
    except IOError as e:
        print(f"Error writing to file {filepath}: {e}")
        return False
    except TypeError as e:
        print(
            f"Data type error during serialization: {e}. Check if data is JSON serializable."
        )
        return False


def read_json(filepath: str) -> Union[JsonData, None]:
    """
    Reads and parses data from a specified JSON file path.

    Args:
        filepath: The path to the input JSON file.

    Returns:
        The Python data structure (dict or list) loaded from the file,
        or None if an error occurred.
    """
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return None

    try:
        # Open file in read mode ('r'), specifying UTF-8 encoding
        with open(filepath, "r", encoding="utf-8") as f:
            # Use json.load to parse the JSON content
            data = json.load(f)
            print(f"Successfully read data from {filepath}")
            return data
    except json.JSONDecodeError as e:
        print(
            f"Error decoding JSON from {filepath}. File might be empty or corrupted: {e}"
        )
        return None
    except IOError as e:
        print(f"Error reading file {filepath}: {e}")
        return None

In [4]:
DATASET_IDS = [
    "OpenHust/vietnamese-summarization",
    "HaiLong9901/VietNameseLongTextSum",
    "truongpdd/vietnamese_story",
]

In [9]:
ds_1 = load_dataset(DATASET_IDS[0])
ds_2 = load_dataset(DATASET_IDS[1])

Generating train split: 74564 examples [00:04, 17468.58 examples/s]
Generating test split: 100%|██████████| 500/500 [00:00<00:00, 3358.03 examples/s]


In [25]:
doc_d2 = []
for key in ds_2.keys():
    dataset = ds_2[key]
    for data in dataset:
        _data = f"{data['abstract']} {data['article']}"
        doc_d2.append(_data)

len(doc_d2)

3000

In [4]:
merged_data = read_json(filepath="../data/vn_plain_dataset.json")["document"]

Successfully read data from ../data/vn_plain_dataset.json


## Build Summarization

In [5]:
from pydantic import BaseModel


class SummarizationSchema(BaseModel):
    summarized_document: str
    keywords: List[str]


SYSTEM_PROMPT = """
Bạn là chuyên gia tóm tắt tài liệu.

## ĐẦU RA YÊU CẦU

**TÓM TẮT** (5-7 câu):
1. Giới thiệu chủ đề chính
2. Nội dung/luận điểm quan trọng
3. Kết luận

**TỪ KHÓA** (5-7 từ):
Khái niệm chính, tên riêng, thuật ngữ chuyên ngành

## QUY TẮC
- Chính xác, súc tích
- Từ khóa phải có trong tài liệu gốc
- Định dạng: phân cách bằng dấu phẩy
- Đầu ra là tiếng Việt
"""


@retry(
    retry=retry_if_exception_type(Exception),
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
)
async def asummarize_document(
    llm: AsyncOpenAI,
    document: str,
    model_name: str = "gemini-2.0-flash-lite",
    system_prompt: str = SYSTEM_PROMPT,
) -> Dict:
    res = await llm.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": document},
        ],
        response_format=SummarizationSchema,
    )
    parsed_res = res.choices[0].message.parsed
    return {
        "document": document,
        "summary": parsed_res.summarized_document,
        "keywords": parsed_res.keywords,
    }


@retry(
    retry=retry_if_exception_type(Exception),
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
)
def summarize_document(
    llm: OpenAI,
    document: str,
    model_name: str = "gemini-2.0-flash-lite",
    system_prompt: str = SYSTEM_PROMPT,
) -> Dict:
    res = llm.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": document},
        ],
        response_format=SummarizationSchema,
    )
    parsed_res = res.choices[0].message.parsed
    return {
        "document": document,
        "summary": parsed_res.summarized_document,
        "keywords": parsed_res.keywords,
    }

In [7]:
await summarize_document(llm=client, document=merged_data[10])

2025-10-25 12:49:20,504 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


{'document': 'Theo V Cục N Thú y N , đến nay tổng số N lợn N tiêu hủy A là hơn A 23 . 000 con N và dịch N chủ yếu A xuất hiện V tại các hộ N chăn nuôi V nhỏ lẻ A , điều kiện N vệ sinh V và an toàn A sinh học N không tốt A , chưa xuất hiện V tại các trang trại N quy mô N lớn A . Cục N nhận định V , nguy cơ N dịch V tiếp tục V lan V rộng A là rất cao A . Lực lượng N chức năng N Hà Nội diễn tập V ứng phó N dịch tả V lợn N Châu Phi ngày N 7/3 . Về N nguyên nhân N dịch V lây lan N , Cục trưởng N Thú y N Phạm Văn Đông nói V kết quả N điều tra V bước đầu N xác định V nguyên nhân N chính là V một số người N chăn nuôi V , thương lái N chưa nhận thức V đầy đủ A tính chất N nguy hiểm A của dịch bệnh N , vì lợi ích N kinh tế N trước mắt N nên đã mua bán V , vận chuyển V , giết mổ V , tiêu thụ V lợn N bệnh N , lợn N nghi V mắc V bệnh N . Trong khi N đó , các cơ sở N nhỏ lẻ A có V mật độ N chăn nuôi V cao A , hộ N chăn nuôi V lợn N đan xen V trong khu N dân cư N không thường xuyên A thực hiện V biện

In [ ]:
# import asyncio
# import aiofiles
# import json
# import random
# import logging
# from pathlib import Path
# from typing import List, Dict, Any
# from tqdm.asyncio import tqdm_asyncio

# # Silence noisy logs
# for name in ["httpx", "httpcore", "openai", "asyncopenai", "urllib3"]:
#     logging.getLogger(name).setLevel(logging.WARNING)

# logger = logging.getLogger(__name__)
# logging.basicConfig(level=logging.INFO)


# async def append_jsonl(result: Dict[str, Any], filename: Path):
#     """
#     Appends a single JSON record as one line (JSONL format).
#     This is safe for concurrent writes.
#     """
#     try:
#         async with aiofiles.open(filename, "a", encoding="utf-8") as f:
#             line = json.dumps(result, ensure_ascii=False)
#             await f.write(line + "\n")
#             await f.flush()
#     except Exception as e:
#         logger.warning(f"⚠️ Failed to append to {filename.name}: {e}")


# async def process_all_documents(
#     llm,
#     documents: List[str],
#     model_name: str = "gemini-2.0-flash-lite",
#     output_file: str = "summaries.jsonl",
#     concurrency: int = 5,
# ):
#     """
#     Runs summarization concurrently (limited by `concurrency`),
#     writes each result immediately (JSONL format),
#     and displays progress with tqdm.
#     """
#     output_path = Path(output_file).resolve()
#     failed_path = output_path.with_name("failed_summaries.jsonl")

#     print(f"Starting parallel summarization of {len(documents)} documents...")
#     print(f"Output file: {output_path}")

#     semaphore = asyncio.Semaphore(concurrency)
#     failed_documents: List[Dict[str, Any]] = []

#     async def worker(idx: int, doc: str):
#         async with semaphore:
#             try:
#                 result = await summarize_document(
#                     llm=llm, document=doc, model_name=model_name
#                 )
#                 await append_jsonl(result, output_path)
#             except Exception as e:
#                 failed_entry = {"document": doc, "error": str(e)}
#                 failed_documents.append(failed_entry)
#                 await append_jsonl(failed_entry, failed_path)
#             finally:
#                 await asyncio.sleep(random.uniform(0.05, 0.1))

#     tasks = [worker(i + 1, doc) for i, doc in enumerate(documents)]
#     await tqdm_asyncio.gather(*tasks, total=len(tasks), desc="Summarizing", leave=True)

#     print("\n--- Processing Complete ---")
#     print(f"✅ Results written incrementally to {output_path}")
#     if failed_documents:
#         print(f"⚠️ {len(failed_documents)} failed → {failed_path}")
#     else:
#         print("✅ All documents processed successfully!")

#     return {"failed": failed_documents, "output_file": str(output_path)}

In [6]:
import json
import logging
import random
from pathlib import Path
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ---------- Logging ----------
for name in ["httpx", "httpcore", "openai", "urllib3"]:
    logging.getLogger(name).setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)


# ---------- Utility ----------
def append_jsonl(result: Dict[str, Any], filename: Path):
    """Append a single record to a JSONL file (safe for concurrent writes)."""
    try:
        with open(filename, "a", encoding="utf-8") as f:
            line = json.dumps(result, ensure_ascii=False)
            f.write(line + "\n")
            f.flush()
    except Exception as e:
        logger.warning(f"⚠️ Failed to append to {filename.name}: {e}")


def safe_extract(
    llm: OpenAI,
    item: Dict[str, Any],
    model_name: str,
    idx: int,
    output_path: Path,
    failed_path: Path,
) -> Dict[str, Any]:
    """
    Wrapper for data_extraction with error handling.
    Each document is processed and written immediately.
    """
    try:
        result = summarize_document(
            llm=llm,
            document=item,
            model_name=model_name,
        )
        result["index"] = idx
        append_jsonl(result, output_path)
        return {"index": idx, "status": "success"}

    except Exception as e:
        failed_entry = {
            "index": idx,
            "description": item.get("description", "")[:300],
            "error": str(e),
        }
        append_jsonl(failed_entry, failed_path)
        logger.warning(f"[{idx}] ❌ Extraction failed: {e}")
        return {"index": idx, "status": "failed", "error": str(e)}


def process_all_documents(
    llm: OpenAI,
    dataset: List[Dict[str, Any]],
    model_name: str = "gemini-2.5-flash",
    output_file: str = "data_extraction_results.jsonl",
    max_workers: int = 10,
):
    """
    Run data_extraction() across multiple threads.
    Each item in `dataset` must have a `description` key.
    Results and failures are written incrementally to disk.
    """
    output_path = Path(output_file).resolve()
    failed_path = output_path.with_name("failed_data_extraction.jsonl")

    print(f"Starting multithreaded extraction for {len(dataset)} records...")
    print(f"Output file: {output_path}")

    results, failed = [], []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i, item in enumerate(dataset, start=1):
            delay = random.uniform(0.05, 0.1)
            futures.append(
                executor.submit(
                    safe_extract, llm, item, model_name, i, output_path, failed_path
                )
            )

        # tqdm progress bar for completed tasks
        for future in tqdm(
            as_completed(futures), total=len(futures), desc="Extracting", unit="doc"
        ):
            result = future.result()
            results.append(result)
            if result["status"] == "failed":
                failed.append(result)

    print("\n--- Processing Complete ---")
    print(f"✅ Results written to: {output_path}")
    if failed:
        print(f"⚠️ {len(failed)} failed → {failed_path}")
    else:
        print("✅ All records processed successfully!")

    return {"failed": failed, "output_file": str(output_path)}

In [7]:
selected_data = merged_data[(len(merged_data) // 2) + 1 :]

len(selected_data)

38781

In [25]:
# selected_data = merged_data[len(merged_data) // 2 + 1: ]

# save_json = {
#     "document": selected_data
# }

# save_data_path = "../data/vn_part_dataset_002.json"
# write_json(save_data_path, save_json)

Successfully wrote data to ../data/vn_part_dataset_002.json


True

In [8]:
"""
NOTE: 
    - 25/10/2025: vn_sum_dataset_001 - 19525
        - something wrong with the async => too slow.
"""

output_file = str(DATA_DIR / "vn_sum_dataset_003.jsonl")
try:
    results = process_all_documents(
        llm=client,
        dataset=selected_data,
        model_name="gemini-2.5-flash-lite",
        output_file=output_file,
        max_workers=16,
    )
except KeyboardInterrupt:
    print("\nProcess interrupted by user.")
except Exception as e:
    print(f"An unexpected error occurred during execution: {e}")

Starting multithreaded extraction for 38781 records...
Output file: /home/octoopt/workspace/projects/personal/data_enrichment/data/vn_sum_dataset_003.jsonl


Extracting:   2%|▏         | 768/38781 [01:25<1:10:09,  9.03doc/s]


An unexpected error occurred during execution: 'str' object has no attribute 'get'


In [9]:
from datasets import load_dataset, DatasetDict
from huggingface_hub import HfApi, HfFolder
import pandas as pd
from pathlib import Path

# ---------------- CONFIG ----------------
JSONL_PATH = DATA_DIR / "vn_sum_dataset_003.jsonl"  # your JSONL output file
REPO_ID = "8Opt/vietnamese-summarization-dataset-0003"  # <- change this
SPLIT_RATIO = [0.8, 0.1, 0.1]  # train/val/test

# ---------------- LOAD DATA ----------------
# Hugging Face can directly load JSONL
dataset = load_dataset("json", data_files=str(JSONL_PATH))["train"]
print(f"✅ Loaded {len(dataset)} samples")

# ---------------- CLEAN DATA ----------------
# Optional cleaning — drop incomplete rows
keep_cols = ["document", "summary", "keywords"]
dataset = dataset.filter(lambda x: all(x.get(c) for c in keep_cols))
print(f"🧹 After cleaning: {len(dataset)} samples")

# ---------------- SPLIT DATA ----------------
# Shuffle before split for randomness
dataset = dataset.shuffle(seed=42)

# 80% train, 10% val, 10% test
train_testvalid = dataset.train_test_split(test_size=0.2, seed=42)
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict(
    {
        "train": train_testvalid["train"],
        "validation": test_valid["train"],
        "test": test_valid["test"],
    }
)

print(dataset_dict)
print({k: len(v) for k, v in dataset_dict.items()})

# ---------------- PUSH TO HUB ----------------
# Login (if not already)
HfFolder.save_token(HF_TOKEN)
api = HfApi()

# Create dataset repo if not exist
api.create_repo(REPO_ID, repo_type="dataset", exist_ok=True)

# Push all splits at once
dataset_dict.push_to_hub(REPO_ID)
print(f"🚀 Successfully pushed dataset with 80/10/10 split to:")
print(f"🔗 https://huggingface.co/datasets/{REPO_ID}")

Generating train split: 38627 examples [00:00, 135876.34 examples/s]


✅ Loaded 38627 samples


Filter: 100%|██████████| 38627/38627 [00:01<00:00, 33215.72 examples/s]


🧹 After cleaning: 38627 samples
DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'keywords', 'index'],
        num_rows: 30901
    })
    validation: Dataset({
        features: ['document', 'summary', 'keywords', 'index'],
        num_rows: 3863
    })
    test: Dataset({
        features: ['document', 'summary', 'keywords', 'index'],
        num_rows: 3863
    })
})
{'train': 30901, 'validation': 3863, 'test': 3863}


Creating parquet from Arrow format: 100%|██████████| 2/2 [00:01<00:00,  1.54ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   1%|          |  525kB / 61.4MB,  164kB/s  
















Processing Files (0 / 1)                :   2%|▏         | 1.05MB / 61.4MB,  159kB/s  









Processing Files (0 / 1)                :   3%|▎         | 1.57MB / 61.4MB,  183kB/s  



Processing Files (0 / 1)                :   3%|▎         | 2.10MB / 61.4MB,  223kB/s  



Processing Files (0 / 1)                :   4%|▍         | 2.62MB / 61.4MB,  257kB/s  



Processing Files (0 / 1)                :   6%|▌         | 3.67MB / 61.4MB,  360kB/s  



Processing Files (0 / 1)                :   7%|▋         | 4.20MB / 61.4MB,  411kB/s  



Processing Files (0 / 1)                :   9%|▉         | 5.77MB / 61.4MB,  566kB/s  


Processing Files (0 / 1)                :  12%|█▏        | 7.34MB / 61.4MB,  720kB/s  



Proces

🚀 Successfully pushed dataset with 80/10/10 split to:
🔗 https://huggingface.co/datasets/8Opt/vietnamese-summarization-dataset-0003


## Build NER-POS

In [4]:
datapath = DATA_DIR / "archive"
datafiles = glob.glob(str(datapath / "2025*.json"))

print(datafiles)

['/home/octoopt/workspace/projects/personal/data_enrichment/data/archive/20252210_001_fmppl_bf.json', '/home/octoopt/workspace/projects/personal/data_enrichment/data/archive/20252210_003_fmppl_bf.json', '/home/octoopt/workspace/projects/personal/data_enrichment/data/archive/20252210_002_fmppl_bf.json']


In [11]:
dataset = []

for datafile in datafiles:
    ds = read_json(filepath=datafile)
    for data in ds:
        metadata = data["metadata"]
        description = metadata["description"]
        if not description:
            continue

        _data = {}
        _data["english"] = description
        _data["name"] = data["name"]
        _data["occupation"] = data["mainType"]
        _data["dob"] = data["dateOfBirth"]
        _data["home_place"] = data["homePlace"]
        _data["image_url"] = data["image_url"]
        _data["profile_url"] = data["profile_url"]
        _data["sunSign"] = metadata.get("sunSign")
        dataset.append(_data)


len(dataset)

Successfully read data from /home/octoopt/workspace/projects/personal/data_enrichment/data/archive/20252210_001_fmppl_bf.json
Successfully read data from /home/octoopt/workspace/projects/personal/data_enrichment/data/archive/20252210_003_fmppl_bf.json
Successfully read data from /home/octoopt/workspace/projects/personal/data_enrichment/data/archive/20252210_002_fmppl_bf.json


8367

In [12]:
dataset[0]

{'english': "A young social media star, Charli D'Amelio has millions of followers on Instagram and Tiktok. In her short career till now, she has performed with Jennifer Lopez, announced the release of her book, made her feature film debut in an animated children's film and performed at the NBA All-Star Game.",
 'name': 'Charli Damelio',
 'occupation': 'TikTok Star',
 'dob': '2004-05-01T00:00:00Z',
 'home_place': 'Norwalk, Connecticut, United States',
 'image_url': 'https://www.thefamouspeople.com/profiles/thumbs/charli-damelio-1.jpg',
 'profile_url': 'https://www.thefamouspeople.com/profiles/charli-damelio-56142.php',
 'sunSign': 'Taurus'}

In [14]:
from typing import List
from pydantic import BaseModel
from openai import OpenAI


# ---------- Pydantic Schemas ----------


class NER(BaseModel):
    persons: List[str]
    organizations: List[str]
    locations: List[str]
    dates: List[str]
    others: List[str]


class MultiLingual(BaseModel):
    vietnamese: str
    german: str
    french: str


class EndResponse(BaseModel):
    multilingual: MultiLingual
    ner: NER


# ---------- Vietnamese System Prompt ----------

SYSTEM_PROMPT = """
Bạn là một trợ lý AI hữu ích, có nhiệm vụ **trích xuất thông tin cấu trúc từ đoạn văn bản đầu vào**.

## NHIỆM VỤ

1. **Nhận dạng thực thể tên (NER)**  
   - Xác định và phân loại các thực thể sau trong văn bản:  
     - persons: tên người  
     - organizations: tổ chức, công ty, cơ quan  
     - locations: địa điểm, quốc gia, thành phố  
     - dates: thời gian, năm, tháng, ngày  
     - others: các loại thực thể khác (sự kiện, sản phẩm, v.v.)

2. **Tạo bản dịch / diễn giải đa ngôn ngữ**  
   - Diễn giải hoặc dịch nội dung đầu vào sang 3 ngôn ngữ:
     - Tiếng Việt
     - Tiếng Đức
     - Tiếng Pháp

## QUY TẮC ĐẦU RA

- Phải trả về đúng theo cấu trúc `EndResponse`.
- Không thêm lời giải thích hoặc chú thích ngoài kết quả.
"""


# ---------- Function ----------


def data_extraction(
    llm: OpenAI,
    data: dict,
    model_name: str = "gemini-2.5-flash",
    response_format: BaseModel = EndResponse,
    system_prompt: str = SYSTEM_PROMPT,
) -> EndResponse:
    """
    Gọi mô hình LLM để trích xuất thông tin NER và bản dịch đa ngôn ngữ
    từ nội dung văn bản đầu vào.
    """
    content = data["english"]
    completion = llm.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content},
        ],
        response_format=response_format,
    )
    result = completion.choices[0].message.parsed.model_dump()
    end_result = {**data, **result["multilingual"], "ner": result["ner"]}
    return end_result

In [15]:
result = data_extraction(client, dataset[0])
result

2025-10-26 10:14:58,586 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


{'english': "A young social media star, Charli D'Amelio has millions of followers on Instagram and Tiktok. In her short career till now, she has performed with Jennifer Lopez, announced the release of her book, made her feature film debut in an animated children's film and performed at the NBA All-Star Game.",
 'name': 'Charli Damelio',
 'occupation': 'TikTok Star',
 'dob': '2004-05-01T00:00:00Z',
 'home_place': 'Norwalk, Connecticut, United States',
 'image_url': 'https://www.thefamouspeople.com/profiles/thumbs/charli-damelio-1.jpg',
 'profile_url': 'https://www.thefamouspeople.com/profiles/charli-damelio-56142.php',
 'sunSign': 'Taurus',
 'vietnamese': "Một ngôi sao mạng xã hội trẻ tuổi, Charli D'Amelio, có hàng triệu người theo dõi trên Instagram và Tiktok. Trong sự nghiệp ngắn ngủi của mình cho đến nay, cô đã biểu diễn cùng Jennifer Lopez, công bố phát hành cuốn sách của mình, ra mắt trong một bộ phim hoạt hình dành cho trẻ em và biểu diễn tại trận đấu NBA All-Star.",
 'german': "C

In [16]:
import json
import logging
import random
from pathlib import Path
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI

# ---------- Logging ----------
for name in ["httpx", "httpcore", "openai", "urllib3"]:
    logging.getLogger(name).setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)


# ---------- Utility ----------
def append_jsonl(result: Dict[str, Any], filename: Path):
    """Append a single record to a JSONL file (safe for concurrent writes)."""
    try:
        with open(filename, "a", encoding="utf-8") as f:
            line = json.dumps(result, ensure_ascii=False)
            f.write(line + "\n")
            f.flush()
    except Exception as e:
        logger.warning(f"⚠️ Failed to append to {filename.name}: {e}")


def safe_extract(
    llm: OpenAI,
    item: Dict[str, Any],
    model_name: str,
    idx: int,
    output_path: Path,
    failed_path: Path,
) -> Dict[str, Any]:
    """
    Wrapper for data_extraction with error handling.
    Each document is processed and written immediately.
    """
    try:
        result = data_extraction(
            llm=llm,
            data=item,
            model_name=model_name,
        )
        result["index"] = idx
        append_jsonl(result, output_path)
        return {"index": idx, "status": "success"}

    except Exception as e:
        failed_entry = {
            "index": idx,
            "description": item.get("description", "")[:300],
            "error": str(e),
        }
        append_jsonl(failed_entry, failed_path)
        logger.warning(f"[{idx}] ❌ Extraction failed: {e}")
        return {"index": idx, "status": "failed", "error": str(e)}


# ---------- Main Runner ----------
def process_all_documents(
    llm: OpenAI,
    dataset: List[Dict[str, Any]],
    model_name: str = "gemini-2.5-flash",
    output_file: str = "data_extraction_results.jsonl",
    max_workers: int = 10,
):
    """
    Run data_extraction() across multiple threads.
    Each item in `dataset` must have a `description` key.
    Results and failures are written incrementally to disk.
    """
    output_path = Path(output_file).resolve()
    failed_path = output_path.with_name("failed_data_extraction.jsonl")

    print(f"Starting multithreaded extraction for {len(dataset)} records...")
    print(f"Output file: {output_path}")

    results, failed = [], []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i, item in enumerate(dataset, start=1):
            # Optional: random delay to reduce burst load on API
            delay = random.uniform(0.05, 0.1)
            futures.append(
                executor.submit(
                    safe_extract, llm, item, model_name, i, output_path, failed_path
                )
            )

        for future in as_completed(futures):
            result = future.result()
            results.append(result)
            if result["status"] == "failed":
                failed.append(result)

    print("\n--- Processing Complete ---")
    print(f"✅ Results written to: {output_path}")
    if failed:
        print(f"⚠️ {len(failed)} failed → {failed_path}")
    else:
        print("✅ All records processed successfully!")

    return {"failed": failed, "output_file": str(output_path)}

In [17]:
try:
    process_all_documents(
        llm=client,
        dataset=dataset,
        model_name="gemini-2.5-flash-lite",
        output_file="../data/famous_people_wiki_0001.jsonl",
        max_workers=16,
    )
except Exception as e:
    print(e)

Starting multithreaded extraction for 8367 records...
Output file: /home/octoopt/workspace/projects/personal/data_enrichment/data/famous_people_wiki_0001.jsonl


2025-10-26 10:26:19,371 - WARNING - [3762] ❌ Extraction failed: 'NoneType' object has no attribute 'model_dump'
2025-10-26 10:32:15,139 - WARNING - [6900] ❌ Extraction failed: 'NoneType' object has no attribute 'model_dump'



--- Processing Complete ---
✅ Results written to: /home/octoopt/workspace/projects/personal/data_enrichment/data/famous_people_wiki_0001.jsonl
⚠️ 2 failed → /home/octoopt/workspace/projects/personal/data_enrichment/data/failed_data_extraction.jsonl


In [9]:
from datasets import load_dataset, DatasetDict
from huggingface_hub import HfApi, HfFolder
import pandas as pd
from pathlib import Path

# ---------------- CONFIG ----------------
JSONL_PATH = DATA_DIR / "famous_people_wiki_0001.jsonl"  # your JSONL output file
REPO_ID = "8Opt/famous-people-wiki-0001"  # <- change this
SPLIT_RATIO = [0.8, 0.1, 0.1]  # train/val/test

# ---------------- LOAD DATA ----------------
# Hugging Face can directly load JSONL
dataset = load_dataset("json", data_files=str(JSONL_PATH))["train"]
print(f"✅ Loaded {len(dataset)} samples")


# ---------------- CLEAN DATA ----------------
keep_cols = [
    "description",
    "name",
    "occupation",
    "dob",
    "home_place",
    "image_url",
    "profile_url",
    "sunSign",
    "vietnamese",
    "german",
    "french",
    "ner",
]
dataset = dataset.filter(lambda x: all(x.get(c) for c in keep_cols))
print(f"🧹 After cleaning: {len(dataset)} samples")

# ---------------- SPLIT DATA ----------------
# Shuffle before split for randomness
dataset = dataset.shuffle(seed=42)

# 80% train, 10% val, 10% test
train_testvalid = dataset.train_test_split(test_size=0.2, seed=42)
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict(
    {
        "train": train_testvalid["train"],
        "validation": test_valid["train"],
        "test": test_valid["test"],
    }
)

print(dataset_dict)
print({k: len(v) for k, v in dataset_dict.items()})

# ---------------- PUSH TO HUB ----------------
# Login (if not already)
HfFolder.save_token(HF_TOKEN)
api = HfApi()

# Create dataset repo if not exist
api.create_repo(REPO_ID, repo_type="dataset", exist_ok=True)

# Push all splits at once
dataset_dict.push_to_hub(REPO_ID)
print(f"🚀 Successfully pushed dataset with 80/10/10 split to:")
print(f"🔗 https://huggingface.co/datasets/{REPO_ID}")

✅ Loaded 8365 samples
🧹 After cleaning: 0 samples
DatasetDict({
    train: Dataset({
        features: ['english', 'name', 'occupation', 'dob', 'home_place', 'image_url', 'profile_url', 'sunSign', 'vietnamese', 'german', 'french', 'ner', 'index'],
        num_rows: 0
    })
    validation: Dataset({
        features: ['english', 'name', 'occupation', 'dob', 'home_place', 'image_url', 'profile_url', 'sunSign', 'vietnamese', 'german', 'french', 'ner', 'index'],
        num_rows: 0
    })
    test: Dataset({
        features: ['english', 'name', 'occupation', 'dob', 'home_place', 'image_url', 'profile_url', 'sunSign', 'vietnamese', 'german', 'french', 'ner', 'index'],
        num_rows: 0
    })
})
{'train': 0, 'validation': 0, 'test': 0}


Creating parquet from Arrow format: 0ba [00:00, ?ba/s]:00<?, ? shards/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████| 4.78kB / 4.78kB,   ???B/s  


Processing Files (1 / 1)                : 100%|██████████| 4.78kB / 4.78kB,  0.00B/s  
New Data Upload                         : |          |  0.00B /  0.00B,  0.00B/s  
                                        : 100%|██████████| 4.78kB / 4.78kB            
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.65s/ shards]
Creating parquet from Arrow format: 0ba [00:00, ?ba/s]:00<?, ? shards/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████| 4.78kB / 4.78kB,   ???B/s  

Processing Files (1 / 1)                : 100%|██████████| 4.78kB / 4.78kB,  0.00B/s  
New Data Upload                         : |          |  0.00B /  0.00B,  0.00B/s  
 

🚀 Successfully pushed dataset with 80/10/10 split to:
🔗 https://huggingface.co/datasets/8Opt/famous-people-wiki-0001


## Building STS

In [4]:
dataset_ids = [
    # "mteb/sts13-sts",
    # "mteb/sts14-sts",
    # "mteb/sts15-sts",
    # "mteb/sts16-sts",
    # "mteb/mteb-human-sts12-sts",
    # "mteb/biosses-sts"
    "sentence-transformers/agnews"
]


# NOTE: only focus on sentence1, sentence2, score

In [5]:
dataset = []

for d_id in dataset_ids:
    _dataset = load_dataset(d_id)
    _dataset = _dataset['train'].to_dict()

    for title, description in zip(_dataset['title'], _dataset['description']):
        article = f"{title}: {description}"
        dataset.append(article)

In [6]:
len(dataset) // 12

96478

In [ ]:
# dataset = {"sentence1": [], "sentence2": [], "score": []}


# for d_id in dataset_ids:
#     _dataset = load_dataset(d_id)
#     ds = _dataset.get("test")
#     if not ds:
#         continue
#     ds = ds.to_dict()
#     dataset["sentence1"] += ds["sentence1"]
#     dataset["sentence2"] += ds["sentence2"]
#     dataset["score"] += ds["score"]


# len(dataset)

3

In [9]:
len(dataset['sentence1'])

9586

In [ ]:
# from googletrans import Translator


# translator = Translator()


# async def translate_to_vn(sentence: str, translator: Translator) -> str:
#     "translate en to vn"
#     res = await translator.translate(text=sentence, src="en", dest="vi")
#     return res.text

In [7]:
# ---------- Vietnamese System Prompt ----------
SYSTEM_PROMPT = """
Bạn là một trợ lý AI hữu ích, có nhiệm vụ:
1. Dịch đoạn văn tiếng Anh sang tiếng Việt.
2. Tạo **anchor** (câu gốc đã được dịch) và **negative sample** (câu mang nghĩa trái ngược hoặc khác ngữ cảnh so với anchor).
3. Trích xuất **từ khóa chính (keywords)** trong đoạn văn bản để biểu diễn nội dung cốt lõi.

## HƯỚNG DẪN CHI TIẾT

- **Anchor**: Là bản dịch tiếng Việt trung thực, tự nhiên, giữ nguyên ý nghĩa của câu gốc.
- **Negative**: Là câu tiếng Việt có ý nghĩa khác biệt hoặc mâu thuẫn với anchor, nhưng vẫn hợp ngữ pháp và tự nhiên.
- **Keywords**: Danh sách 3–7 từ khóa quan trọng thể hiện nội dung chính của anchor (viết thường, không chứa dấu câu).

## QUY TẮC ĐẦU RA

- Trả về dữ liệu đúng cấu trúc `NLI_Triplet`.
- Không thêm giải thích, nhận xét hoặc nội dung ngoài kết quả.
- Câu viết bằng tiếng Việt, có dấu đầy đủ, tự nhiên.
"""

class NLI_Triplet(BaseModel):
    positive: str
    negative: str
    anchors: str
    keywords: list[str]

# ---------- Core Extraction Function ----------
def data_extraction(
    llm: OpenAI,
    data: str,
    model_name: str = "gemini-2.5-flash-lite",
    response_format: BaseModel = NLI_Triplet,
    system_prompt: str = SYSTEM_PROMPT,
) -> dict:
    """Translate two English sentences to Vietnamese and return structured output."""

    content = f"""
    Đây là hai câu cần dịch sang tiếng Việt:

    Câu: {data}
    Nhiệm vụ của bạn:
    - Dịch các câu này sang tiếng Việt.
    - Dựa trên ngữ nghĩa, hãy tạo ra:
      (1) Một **anchor** tiếng Việt thể hiện trung thực ý nghĩa của câu đầu tiên.
      (2) Một **negative sample** tiếng Việt có ý nghĩa khác biệt hoặc trái ngược với anchor.
      (3) Danh sách **từ khóa (keywords)** mô tả nội dung chính.
      (4) Các câu nên có cùng độ dài.
    """

    completion = llm.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": content.strip()},
        ],
        response_format=response_format,
    )

    result = completion.choices[0].message.parsed.model_dump()
    return result

In [12]:
import json
import logging
import random
from pathlib import Path
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from rich.live import Live
from rich.console import Console, Group
from rich.table import Table
from rich.panel import Panel
from rich.progress import Progress, BarColumn, TextColumn, TimeElapsedColumn
from rich.layout import Layout
from rich.box import ROUNDED, HEAVY, DOUBLE

# ---------- Logging ----------
for name in ["httpx", "httpcore", "openai", "urllib3"]:
    logging.getLogger(name).setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

console = Console()



# ---------- Utility ----------
def append_jsonl(result: Dict[str, Any], filename: Path):
    """Append a single record to a JSONL file (safe for concurrent writes)."""
    try:
        with open(filename, "a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()
    except Exception as e:
        logger.warning(f"⚠️ Failed to append to {filename.name}: {e}")


def safe_extract(
    llm: OpenAI,
    item: Dict[str, Any] | str,
    func: callable,
    model_name: str,
    idx: int,
    output_path: Path,
    failed_path: Path,
) -> Dict[str, Any]:
    """Wrapper for data_extraction with error handling."""
    try:
        result = func(llm=llm, data=item, model_name=model_name)
        result["index"] = idx
        append_jsonl(result, output_path)
        return {"index": idx, "status": "success"}
    except Exception as e:
        failed_entry = {
            "index": idx,
            "sentence1": item.get("sentence1", "")[:100],
            "error": str(e),
        }
        append_jsonl(failed_entry, failed_path)
        return {"index": idx, "status": "failed", "error": str(e)}


# ---------- Main Runner ----------
def process_all_documents(
    llm: OpenAI,
    dataset: List[Dict[str, Any]] | List[str],
    func: callable,
    model_name: str = "gemini-2.5-flash",
    output_file: str = "data_extraction_results.jsonl",
    max_workers: int = 10,
):
    """Run data_extraction() across multiple threads with a styled Rich dashboard."""
    output_path = Path(output_file).resolve()
    failed_path = output_path.with_name("failed_data_extraction.jsonl")

    console.print(f"[bold cyan]Starting extraction for {len(dataset)} records...[/bold cyan]")
    console.print(f"[bold white]Output file:[/bold white] {output_path}\n")

    results, failed = [], []

    # Progress bar setup
    progress = Progress(
        TextColumn("[bold blue]{task.description}"),
        BarColumn(),
        TextColumn("{task.completed}/{task.total}"),
        TimeElapsedColumn(),
        console=console,
        expand=True,
    )
    task_id = progress.add_task("Processing", total=len(dataset))

    # Log table setup
    log_table = Table(title="Extraction Log", box=ROUNDED, show_lines=False, border_style="cyan")
    log_table.add_column("Index", justify="right", style="bold white")
    log_table.add_column("Status", style="bold")
    log_table.add_column("Message", overflow="fold", style="dim")

    try:
        with Live(
            Panel.fit("Initializing...", border_style="bright_blue", title="Extraction Progress"),
            refresh_per_second=5,
            console=console,
        ) as live:
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [
                    executor.submit(
                        safe_extract, llm, item, func, model_name, i + 1, output_path, failed_path
                    )
                    for i, item in enumerate(dataset)
                ]

                for future in as_completed(futures):
                    result = future.result()
                    results.append(result)

                    if result["status"] == "failed":
                        failed.append(result)
                        log_table.add_row(
                            str(result["index"]),
                            "[red]❌ Failed",
                            result.get("error", ""),
                        )
                    else:
                        log_table.add_row(
                            str(result["index"]),
                            "[green]✅ Success",
                            "Processed successfully",
                        )

                    progress.advance(task_id)
                    summary = f"[green]✅ {len(results) - len(failed)}[/green] | [red]❌ {len(failed)}[/red]"

                    # --- Create bounded panels ---
                    progress_panel = Panel(
                        progress,
                        title=f"Progress — {summary}",
                        border_style="bright_green",
                        box=DOUBLE,
                        padding=(1, 2),
                        style="on default"
                    )

                    log_panel = Panel(
                        log_table,
                        title="Recent Logs",
                        border_style="bright_cyan",
                        box=ROUNDED,
                        padding=(0, 1),
                            style="on default", 
                    )

                    dashboard = Panel(
                        Group(progress_panel, log_panel),
                        title="[bold white on blue] Extraction Dashboard [/]",
                        border_style="bright_blue",
                        padding=(1, 2),
                        box=HEAVY,
                        style="on default",
                    )

                    live.update(dashboard)

        console.print("\n[bold green]--- Processing Complete ---[/bold green]")
        console.print(f"✅ Results written to: {output_path}")
        if failed:
            console.print(f"[bold red]⚠️ {len(failed)} failed → {failed_path}[/bold red]")
        else:
            console.print("[bold green]✅ All records processed successfully![/bold green]")

    except Exception as e:
        console.print(f"\n[bold red]❌ An unexpected error occurred: {e}[/bold red]")
        logger.exception("Fatal error in process_all_documents")
        console.print(f"Processed so far: {len(results)} | Failed: {len(failed)}")
        console.print(f"Partial results saved to: {output_path}")

    return {"failed": failed, "output_file": str(output_path)}



In [13]:
dest = "../data/vn-nli-triplet-0001.jsonl"

try:
    process_all_documents(
        llm=client,
        dataset=dataset[4696: len(dataset) // 12],
        func=data_extraction,
        model_name="gemini-2.5-flash-lite",
        output_file=dest,
        max_workers=16,
    )
except Exception as e:
    console.print(f"[bold red]❌ Fatal error in main: {e}[/bold red]")


Starting extraction for 91782 records...

Output file: /home/octoopt/workspace/projects/personal/data_enrichment/data/vn-nli-triplet-0001.jsonl

Output()

KeyboardInterrupt: 

In [1]:
# write_json(filepath=str(DATA_DIR / "vn-sts-0001.json"), data=translated_ds)

In [2]:
from datasets import load_dataset, DatasetDict
from huggingface_hub import HfApi, HfFolder
import pandas as pd
from pathlib import Path
import os 

from dotenv import  load_dotenv

HF_TOKEN = os.getenv("HF_TOKEN")
# ---------------- CONFIG ----------------
JSONL_PATH =  "../data/vn-nli-triplet-0001.jsonl"  # your JSONL output file
REPO_ID = "8Opt/vn-nli-triplet"  # <- change this
SPLIT_RATIO = [0.7, 0.1, 0.2]  # train/val/test

# ---------------- LOAD DATA ----------------
# Hugging Face can directly load JSONL
dataset = load_dataset("json", data_files=str(JSONL_PATH))["train"]
print(f"✅ Loaded {len(dataset)} samples")


# ---------------- CLEAN DATA ----------------
keep_cols = ['positive', 'negative', 'anchors', 'keywords']
dataset = dataset.filter(lambda x: all(x.get(c) for c in keep_cols))
print(f"🧹 After cleaning: {len(dataset)} samples")

# ---------------- SPLIT DATA ----------------
# Shuffle before split for randomness
dataset = dataset.shuffle(seed=1026)

# 80% train, 10% val, 10% test
train_testvalid = dataset.train_test_split(test_size=0.2, seed=1026)
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=1026)

dataset_dict = DatasetDict(
    {
        "train": train_testvalid["train"],
        "validation": test_valid["train"],
        "test": test_valid["test"],
    }
)

print(dataset_dict)
print({k: len(v) for k, v in dataset_dict.items()})

# ---------------- PUSH TO HUB ----------------
# Login (if not already)
HfFolder.save_token(HF_TOKEN)
api = HfApi()

# Create dataset repo if not exist
api.create_repo(REPO_ID, repo_type="dataset", exist_ok=True)

# Push all splits at once
dataset_dict.push_to_hub(REPO_ID)
print(f"🚀 Successfully pushed dataset with 80/10/10 split to:")
print(f"🔗 https://huggingface.co/datasets/{REPO_ID}")

✅ Loaded 19039 samples


Filter:   0%|          | 0/19039 [00:00<?, ? examples/s]

🧹 After cleaning: 19039 samples
DatasetDict({
    train: Dataset({
        features: ['positive', 'negative', 'anchors', 'keywords', 'index'],
        num_rows: 15231
    })
    validation: Dataset({
        features: ['positive', 'negative', 'anchors', 'keywords', 'index'],
        num_rows: 1904
    })
    test: Dataset({
        features: ['positive', 'negative', 'anchors', 'keywords', 'index'],
        num_rows: 1904
    })
})
{'train': 15231, 'validation': 1904, 'test': 1904}


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  60%|######    |  525kB /  873kB            

README.md:   0%|          | 0.00/600 [00:00<?, ?B/s]

🚀 Successfully pushed dataset with 80/10/10 split to:
🔗 https://huggingface.co/datasets/8Opt/vn-nli-triplet


In [4]:
sentence = "Bác sĩ bây giờ có thể thản nhiên báo tin bệnh nhân bị ung thư"

word_tokenize(sentence)

['Bác sĩ',
 'bây giờ',
 'có thể',
 'thản nhiên',
 'báo',
 'tin',
 'bệnh nhân',
 'bị',
 'ung thư']

In [2]:
word_tokenize(sentence, format="text")

'Bác_sĩ bây_giờ có_thể thản_nhiên báo tin bệnh_nhân bị ung_thư'

In [6]:
import os
from datasets import load_dataset
from dotenv import load_dotenv


load_dotenv()


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

In [7]:
from openai import OpenAI

# 1. Initialize the OpenAI client, but configure it for Gemini
client = OpenAI(
    # Replace "YOUR_GEMINI_API_KEY" with your actual Gemini API key
    api_key=GEMINI_API_KEY,
    # This base_url routes the request to the Gemini API's compatibility layer
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# 2. Make a chat completion request as you normally would,
# but specify a compatible Gemini model (e.g., "gemini-2.5-flash")
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain to me how AI works"},
    ],
)

print(response.choices[0].message.content)

At its core, **AI (Artificial Intelligence)** is about enabling machines to **simulate human intelligence**. This doesn't mean they think exactly like us, but rather they can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, understanding language, and recognizing patterns.

The most common and impactful way AI works today, especially in what's called **Machine Learning (a subset of AI)**, is through **learning from data**. Instead of being explicitly programmed for every single scenario, AI models learn to make predictions or decisions based on patterns they identify in vast amounts of information.

Let's break it down into simpler steps:

### The Core Idea: Learning from Experience

Imagine you want to teach a child to identify cats.
1.  **You show them many pictures:** Some pictures are of cats, some are of dogs, birds, or other animals.
2.  **You label them:** "This is a cat," "This is a dog," "This is a cat."
3.  **The chi

In [9]:
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI(
    # Replace "YOUR_GEMINI_API_KEY" with your actual Gemini API key
    api_key=GEMINI_API_KEY,
    # This base_url routes the request to the Gemini API's compatibility layer
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


completion = client.beta.chat.completions.parse(
    model="gemini-2.0-flash",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "John and Susan are going to an AI conference on Friday.",
        },
    ],
    response_format=CalendarEvent,
)

print(completion.choices[0].message.parsed)

name='AI conference' date='Friday' participants=['John', 'Susan']
